In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql.window import Window

### Create Schema

In [0]:
spark.sql('create schema if not exists test01.gold')

---
### Create dim, fact Table

#### dim_cust

In [0]:
df_silver_crm_cust_info = spark.table('test01.silver.crm_cust_info')
df_silver_erp_cust_az12 = spark.table('test01.silver.erp_cust_az12')
df_silver_erp_loc_a101 = spark.table('test01.silver.erp_loc_a101')

df_gold_dim_cust = (
    df_silver_crm_cust_info.alias('c').join(
        df_silver_erp_cust_az12.alias('e'), 
        on = col('c.cst_key') == col('e.cid'),
        how = 'left'
    )
    .join(
        df_silver_erp_loc_a101.alias('l'),
        on = col('c.cst_key') == col('l.cid'),
        how = 'left'
    )
    .withColumn(
        'gender',
        when(col('c.cst_gndr') != 'n/a', col('cst_gndr'))
        .otherwise(coalesce(col('e.gen'), lit('n/a')))
    )
    .select(
        col('c.cst_id').alias('customer_id'),
        col('c.cst_key').alias('customer_key'),
        col('c.cst_firstname').alias('first_name'),
        col('c.cst_lastname').alias('last_name'),
        col('c.cst_marital_status').alias('marital_status'),
        col('gender'),
        col('e.bdate').alias('birth_date'),
        col('l.cntry').alias('country'),
        col('c.cst_create_date').alias('create_date')
    )
)

df_gold_dim_cust.display()

#### dim_prd

In [0]:
df_silver_crm_prd_info = spark.table('test01.silver.crm_prd_info')
df_silver_erp_px_cat_g1v2 = spark.table('test01.silver.erp_px_cat_g1v2')

df_gold_dim_prd = (
    df_silver_crm_prd_info.alias('p').join(
        df_silver_erp_px_cat_g1v2.alias('c'),
        on = col('p.cat_id') == col('c.id'),
        how = 'left'
    )
    .select(
        col('p.prd_id').alias('product_id'),
        col('p.prd_key').alias('product_key'),
        col('p.cat_id').alias('product_category'),
        col('p.prd_nm').alias('product_name'),
        col('c.cat').alias('category'),
        col('c.subcat').alias('sub_category'),
        col('c.maintenance').alias('maintenance'),
        col('p.prd_cost').alias('cost'),
        col('p.prd_line').alias('product_line'),
        col('p.prd_start_dt').alias('start_date')
    )
    .filter(col('p.prd_end_dt').isNull())
)

df_gold_dim_prd.display()

#### fact_sales

In [0]:
df_silver_crm_sales_details = spark.table('test01.silver.crm_sales_details')

df_gold_fact_sales = (
    df_silver_crm_sales_details.alias('s').join(
        df_gold_dim_cust.alias('c'),
        on = col('s.sls_cst_id') == col('c.customer_id')
    ).join(
        df_gold_dim_prd.alias('p'),
        on = col('s.sls_prd_key') == col('p.product_key'),
        how = 'left'
    )
    .select(
        col('s.sls_ord_num').alias('order_number'),
        col('c.customer_id'),
        col('p.product_key'),
        col('s.sls_order_dt').alias('order_date'),
        col('s.sls_ship_dt').alias('ship_date'),
        col('s.sls_due_dt').alias('due_date'),
        col('s.sls_price').alias('price'),
        col('s.sls_quantity').alias('quantity'),
        col('s.sls_sales').alias('sales')
    )
)

df_gold_fact_sales.display()

### Write

In [0]:
df_gold_dim_cust.write.format('delta').mode('overwrite').option('overwriteSchema', True).saveAsTable('test01.gold.dim_cust')

df_gold_dim_prd.write.format('delta').mode('overwrite').option('overwriteSchema', True).saveAsTable('test01.gold.dim_prd')

df_gold_fact_sales.write.format('delta').mode('overwrite').option('overwriteSchema', True).saveAsTable('test01.gold.fact_sales')

In [0]:
spark.table('test01.gold.dim_cust').display()